In [31]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_openml
from scipy.stats import mode

In [32]:
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)
y = y.astype(int) 
X = X / 255.0

In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [34]:
pca = PCA(n_components=50, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

In [35]:
kmeans = KMeans(n_clusters=10, n_init=10, init="k-means++", max_iter=300, random_state=42)
kmeans.fit(X_train_pca)

KMeans(n_clusters=10, n_init=10, random_state=42)

In [36]:
cluster_labels = {}
for i in range(10):
    mask = (kmeans.labels_ == i)
    if np.any(mask):
        cluster_labels[i] = mode(y_train[mask], keepdims=False).mode

In [38]:
pseudo_labels = np.array([cluster_labels[c] for c in kmeans.labels_])

In [39]:
clf = LogisticRegression(max_iter=200, solver="lbfgs", multi_class="multinomial", n_jobs=-1)
clf.fit(X_train_pca, pseudo_labels)

c:\Users\Mezon\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogisticRegression(max_iter=200, multi_class='multinomial', n_jobs=-1)

In [40]:
y_pred = clf.predict(X_test_pca)
print("KMeans + Logistic Regression (pseudo-label) Accuracy:", accuracy_score(y_test, y_pred))

KMeans + Logistic Regression (pseudo-label) Accuracy: 0.5824285714285714


In [41]:
clf_real = LogisticRegression(max_iter=200, solver="lbfgs", multi_class="multinomial", n_jobs=-1)
clf_real.fit(X_train_pca, y_train)

y_pred_real = clf_real.predict(X_test_pca)
print("Only Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_real))

c:\Users\Mezon\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Only Logistic Regression Accuracy: 0.9069285714285714
